# T2.2 Evaluation Notebook

Run MRR@5 and Recall@5 on `data/gold_matches.csv`, then inspect the 3 confusion cases.

To reproduce:
```bash
pip install -r requirements.txt
python src/generate_data.py
```
then run this notebook top-to-bottom.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from evaluate import evaluate, print_report
results = evaluate(k=5)
print_report(results)

Tenders: 40   Profiles: 10   k=5
MRR@5:     0.933
Recall@5:  0.933
Index build: 24.2 ms   Avg rank: 0.9 ms   Total: 33.1 ms

Per-profile results:
  Profile Sector       Lang      RR  Recall  Hits
  P01     fintech      fr     1.000   1.000  T016,T019,T033
  P02     cleantech    fr     1.000   0.667  T004,T005
  P03     wastetech    en     1.000   1.000  T006,T020,T039
  P04     cleantech    en     1.000   1.000  T005,T031,T034
  P05     edtech       en     1.000   1.000  T009,T017,T037
  P06     edtech       fr     0.333   1.000  T009,T024,T037
  P07     healthtech   fr     1.000   1.000  T007,T011,T028
  P08     healthtech   fr     1.000   1.000  T007,T008,T028
  P09     agritech     en     1.000   1.000  T010,T022,T038
  P10     agritech     fr     1.000   0.667  T001,T014

3 confusion cases (lowest recall first):
  P02 (cleantech/fr): predicted=['T004', 'T035', 'T034', 'T005', 'T023']  gold=['T031', 'T004', 'T005']  hits=['T004', 'T005']
  P10 (agritech/fr): predicted=['T001', 'T027

## Summary table

In [2]:
import pandas as pd

summary = pd.DataFrame([{
    'Metric': 'MRR@5', 'Value': f"{results['mrr_at_k']:.3f}"
}, {
    'Metric': 'Recall@5', 'Value': f"{results['recall_at_k']:.3f}"
}, {
    'Metric': 'Tenders', 'Value': results['n_tenders']
}, {
    'Metric': 'Profiles', 'Value': results['n_profiles']
}, {
    'Metric': 'End-to-end (ms)', 'Value': results['total_end_to_end_ms']
}])
summary

,Metric,Value
0,MRR@5,0.933
1,Recall@5,0.933
2,Tenders,40
3,Profiles,10
4,End-to-end (ms),33.1


## Per-profile breakdown

In [3]:
df = pd.DataFrame(results['per_profile'])
df[['profile_id', 'sector', 'country', 'primary_lang', 'rr', 'recall', 'predicted', 'gold', 'hits']]

,profile_id,sector,country,primary_lang,rr,recall,predicted,gold,hits
0,P01,fintech,Kenya,fr,1.000000,1.000000,"[T016, T019, T033, T032, T030]","[T019, T016, T033]","[T016, T019, T033]"
1,P02,cleantech,DRC,fr,1.000000,0.666667,"[T004, T035, T034, T005, T023]","[T031, T004, T005]","[T004, T005]"
2,P03,wastetech,Senegal,en,1.000000,1.000000,"[T039, T006, T020, T010, T031]","[T039, T006, T020]","[T006, T020, T039]"
3,P04,cleantech,Ethiopia,en,1.000000,1.000000,"[T034, T005, T023, T031, T004]","[T031, T005, T034]","[T005, T031, T034]"
4,P05,edtech,Senegal,en,1.000000,1.000000,"[T009, T037, T024, T017, T025]","[T037, T009, T017]","[T009, T017, T037]"
5,P06,edtech,Rwanda,fr,0.333333,1.000000,"[T030, T013, T009, T037, T024]","[T037, T009, T024]","[T009, T024, T037]"
6,P07,healthtech,Senegal,fr,1.000000,1.000000,"[T028, T007, T008, T011, T003]","[T028, T007, T011]","[T007, T011, T028]"
7,P08,healthtech,DRC,fr,1.000000,1.000000,"[T028, T007, T003, T026, T008]","[T008, T028, T007]","[T007, T008, T028]"
8,P09,agritech,Rwanda,en,1.000000,1.000000,"[T010, T015, T038, T029, T022]","[T038, T010, T022]","[T010, T022, T038]"
9,P10,agritech,Senegal,fr,1.000000,0.666667,"[T001, T027, T014, T040, T018]","[T001, T015, T014]","[T001, T014]"


## Three confusion cases

Profiles where the ranker most disagreed with the expert labels. These are the cases to
discuss in the Live Defence — each gets a short diagnosis below the table.

In [4]:
cdf = pd.DataFrame(results['confusion_cases'])
cdf[['profile_id', 'sector', 'primary_lang', 'rr', 'recall', 'predicted', 'gold', 'hits']]

,profile_id,sector,primary_lang,rr,recall,predicted,gold,hits
0,P02,cleantech,fr,1.000000,0.666667,"[T004, T035, T034, T005, T023]","[T031, T004, T005]","[T004, T005]"
1,P10,agritech,fr,1.000000,0.666667,"[T001, T027, T014, T040, T018]","[T001, T015, T014]","[T001, T014]"
2,P06,edtech,fr,0.333333,1.000000,"[T030, T013, T009, T037, T024]","[T037, T009, T024]","[T009, T024, T037]"


### Diagnosis

For each confusion case, the miss is not sector confusion (all predictions are in-sector)
but **ranking refinement**: two or more tenders share sector + budget band and the ranker
can't cleanly separate them from lexical overlap alone.

Single feature that would demote the worst miss: **past-funding-to-budget-target ratio** as
a hard gate above 10x — currently it decays smoothly, which lets big-ticket tenders float
into the top-5 for early-stage profiles.